In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
import os

load_dotenv()
MINIML_MODEL_PATH = os.getenv("MINIML_MODEL_PATH")

# Usage
embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)

class ContextAwareSemanticChunker:
    def __init__(self, embeddings_model, similarity_threshold=0.65, window_size=3):
        self.embeddings = embeddings_model
        self.similarity_threshold = similarity_threshold
        self.window_size = window_size  # Look at multiple sentences for context
    
    def split_into_sentences(self, text):
        """Split text into sentences"""
        text = text.replace('\n', ' ')
        sentences = []
        for sent in text.split('. '):
            if sent.strip():
                sentences.append(sent.strip() + '.')
        return sentences
    
    def get_embeddings_batch(self, sentences):
        """Get embeddings for multiple sentences"""
        return [self.embeddings.embed_query(sent) for sent in sentences]
    
    def calculate_context_similarity(self, current_chunk_embs, candidate_emb, sentences_around=None):
        """Calculate similarity with context, not just last sentence"""
        if not current_chunk_embs:
            return 1.0
        
        # Compare with multiple sentences from the chunk for context
        # Use last N sentences from current chunk
        context_embs = current_chunk_embs[-min(self.window_size, len(current_chunk_embs)):]
        
        # Calculate similarities with context sentences
        similarities = []
        for ctx_emb in context_embs:
            sim = cosine_similarity([ctx_emb], [candidate_emb])[0][0]
            similarities.append(sim)
        
        # Use different aggregation strategies
        max_similarity = max(similarities)  # Best match with any context sentence
        avg_similarity = np.mean(similarities)  # Average similarity
        
        # You can also check if topic has shifted completely
        if max_similarity < self.similarity_threshold:
            return 0  # Force split if no good match with context
        
        return avg_similarity
    
    def semantic_chunking(self, text):
        """Main chunking logic with context awareness"""
        sentences = self.split_into_sentences(text)
        
        if len(sentences) <= 2:
            return [text]
        
        # Get embeddings for all sentences
        embeddings_list = self.get_embeddings_batch(sentences)
        
        chunks = []
        current_chunk = [sentences[0]]
        current_embeddings = [embeddings_list[0]]
        
        for i in range(1, len(sentences)):
            current_emb = embeddings_list[i]
            
            # Calculate similarity with context
            context_similarity = self.calculate_context_similarity(
                current_embeddings, current_emb
            )
            
            # Also check if this sentence starts a completely new topic
            # by comparing with the first sentence of the chunk
            first_emb = current_embeddings[0]
            similarity_with_first = cosine_similarity([first_emb], [current_emb])[0][0]
            
            # Decision logic
            if context_similarity >= self.similarity_threshold and similarity_with_first >= 0.5:
                # Still on the same topic
                current_chunk.append(sentences[i])
                current_embeddings.append(current_emb)
            else:
                # Topic shift detected - create new chunk
                if current_chunk:
                    chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
                current_embeddings = [current_emb]
        
        # Add the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        
        return chunks


C:\Users\scientist\AppData\Local\Temp\ipykernel_11156\871942861.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=MINIML_MODEL_PATH)
D:\NewAgentic\RAG_Terms\venv_chunk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4957.01it/s]


In [2]:
text = """
Python is a programming language. It is widely used in AI.
Machine learning is a subset of AI. AI is artificial intelligence. It focuses on AI models.
The stock market crashed yesterday. Investors are worried.
"""

chunker = ContextAwareSemanticChunker(
    embeddings_model=embeddings,
    similarity_threshold=0.65,
    window_size=2  # Look at last 2 sentences for context
)

chunks = chunker.semantic_chunking(text)

print("CONTEXT-AWARE CHUNKING RESULTS:")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk}")

CONTEXT-AWARE CHUNKING RESULTS:

Chunk 1:
Python is a programming language.

Chunk 2:
It is widely used in AI.

Chunk 3:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 4:
It focuses on AI models.

Chunk 5:
The stock market crashed yesterday.

Chunk 6:
Investors are worried.


In [3]:
class TopicBasedSemanticChunker:
    def __init__(self, embeddings_model, threshold=0.7):
        self.embeddings = embeddings_model
        self.threshold = threshold
    
    def split_into_sentences(self, text):
        """Split text into sentences"""
        text = text.replace('\n', ' ')
        sentences = []
        for sent in text.split('. '):
            if sent.strip():
                sentences.append(sent.strip() + '.')
        return sentences
    
    def get_topic_embedding(self, sentences):
        """Get embedding representing the topic of multiple sentences"""
        if not sentences:
            return None
        combined_text = " ".join(sentences)
        return self.embeddings.embed_query(combined_text)
    
    def semantic_chunking(self, text):
        """Chunk based on topic consistency"""
        sentences = self.split_into_sentences(text)
        
        if len(sentences) <= 2:
            return [text]
        
        chunks = []
        current_chunk = [sentences[0]]
        current_topic_emb = self.get_topic_embedding(current_chunk)
        
        for i in range(1, len(sentences)):
            # Get embedding for next sentence
            next_emb = self.embeddings.embed_query(sentences[i])
            
            # Calculate similarity with current topic
            similarity = cosine_similarity([current_topic_emb], [next_emb])[0][0]
            
            # Also check if adding this sentence would change the topic significantly
            test_chunk = current_chunk + [sentences[i]]
            test_topic_emb = self.get_topic_embedding(test_chunk)
            topic_change = cosine_similarity([current_topic_emb], [test_topic_emb])[0][0]
            
            if similarity >= self.threshold and topic_change >= 0.8:
                # Sentence fits current topic
                current_chunk.append(sentences[i])
                current_topic_emb = self.get_topic_embedding(current_chunk)
            else:
                # Topic shift detected
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
                current_topic_emb = self.get_topic_embedding(current_chunk)
        
        # Add the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        
        return chunks

# Test Topic-Based Chunker
print("\n" + "="*50)
print("TOPIC-BASED CHUNKING RESULTS")
print("="*50)

topic_chunker = TopicBasedSemanticChunker(
    embeddings_model=embeddings,
    threshold=0.7
)

topic_chunks = topic_chunker.semantic_chunking(text)

for i, chunk in enumerate(topic_chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


TOPIC-BASED CHUNKING RESULTS

Chunk 1:
Python is a programming language.

Chunk 2:
It is widely used in AI.

Chunk 3:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 4:
It focuses on AI models.

Chunk 5:
The stock market crashed yesterday.

Chunk 6:
Investors are worried.


In [4]:
class AdaptiveThresholdChunker:
    def __init__(self, embeddings_model, base_threshold=0.65):
        self.embeddings = embeddings_model
        self.base_threshold = base_threshold
    
    def split_into_sentences(self, text):
        """Split text into sentences"""
        text = text.replace('\n', ' ')
        sentences = []
        for sent in text.split('. '):
            if sent.strip():
                sentences.append(sent.strip() + '.')
        return sentences
    
    def detect_topic_boundary(self, prev_sentences, next_sentence):
        """Detect if there's a clear topic boundary"""
        if len(prev_sentences) < 2:
            return False
        
        # Get embeddings
        prev_emb = self.embeddings.embed_query(" ".join(prev_sentences[-2:]))
        next_emb = self.embeddings.embed_query(next_sentence)
        
        # Calculate similarity
        similarity = cosine_similarity([prev_emb], [next_emb])[0][0]
        
        # Check for keywords indicating topic shift
        topic_shift_indicators = [
            "however", "meanwhile", "in contrast", "on the other hand",
            "the stock market", "meanwhile", "separately", "in other news"
        ]
        
        has_indicator = any(indicator in next_sentence.lower() 
                          for indicator in topic_shift_indicators)
        
        # Adaptive threshold - lower if no indicator, higher if indicator present
        threshold = self.base_threshold - 0.1 if has_indicator else self.base_threshold + 0.1
        
        return similarity < threshold
    
    def semantic_chunking(self, text):
        """Chunk with adaptive thresholds"""
        sentences = self.split_into_sentences(text)
        
        if len(sentences) <= 2:
            return [text]
        
        chunks = []
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            if self.detect_topic_boundary(current_chunk, sentences[i]):
                # Topic boundary detected
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
            else:
                # Same topic
                current_chunk.append(sentences[i])
        
        # Add the last chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        
        return chunks

# Test Adaptive Threshold Chunker
print("\n" + "="*50)
print("ADAPTIVE THRESHOLD CHUNKING RESULTS")
print("="*50)

adaptive_chunker = AdaptiveThresholdChunker(
    embeddings_model=embeddings,
    base_threshold=0.65
)

adaptive_chunks = adaptive_chunker.semantic_chunking(text)

for i, chunk in enumerate(adaptive_chunks):
    print(f"\nChunk {i+1}:\n{chunk}")


ADAPTIVE THRESHOLD CHUNKING RESULTS

Chunk 1:
Python is a programming language. It is widely used in AI.

Chunk 2:
Machine learning is a subset of AI. AI is artificial intelligence.

Chunk 3:
It focuses on AI models. The stock market crashed yesterday.

Chunk 4:
Investors are worried.
